# Explore smoothing values

In [ ]:
from nescafe23_fluctuations import tools
import xarray as xr
import pygenray as pr
import numpy as np
from matplotlib import pyplot as plt
from scipy import signal
import cmocean

%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = 'retina'

## Compute rays through smoothed and un-smoothed profiles

In [ ]:
dss = tools.open_receptions()

In [ ]:
lower_array_depth = []
for k in range(23):
    lower_array_depth.append(float(dss[k].actual_depth[32:].mean()))

lower_array_depth = np.array(lower_array_depth)

In [ ]:
Z_lower_array = np.mean(lower_array_depth)

In [ ]:
zsmooths = np.array([None, 200])
rsmooths = np.array([None, 15000])

In [ ]:
rays = []
eigs = []

for k in range(len(zsmooths)):

    zsmooth = zsmooths[k]
    rsmooth = rsmooths[k]

    # previously used 550z 10000r
    fctd = tools.compute_filtered_fctd_grid(z_smooth=zsmooth, r_smooth=rsmooth)
    
    # shift front location
    front_location = 35e3  # m
    
    fctd = fctd.assign_coords({
        "range": (fctd.range.values + front_location)
    })
    
    c = fctd["c"]
    
    bathy = xr.DataArray(
        np.linspace(4908, 4908, 100),  # flat bottom
        dims=["range"],
        coords={"range": np.linspace(0, 200e3, 100)},
    )
    
    envi = pr.OceanEnvironment2D(sound_speed=c, bathymetry=bathy)


    source_depth = 2300
    receiver_range = 153e3
    source_range = 0
    receiver_depths = [float(Z_lower_array)]
    launch_angles = np.linspace(-15,15,4000)
    num_range_save = 1000

    print(f'Run {k+1}/{len(zsmooths)}, $z_{{smooth}}$ = {zsmooth} [m]')

    rays.append(pr.shoot_rays(source_depth, source_range, launch_angles, receiver_range, num_range_save, envi))
    eigs.append(pr.find_eigenrays(rays[k], receiver_depths, source_depth, source_range, receiver_range, num_range_save, envi))

In [ ]:
fig = plt.figure(figsize=(10,5))


ks = [0,1]
labels=['No smoothing', 'Z=200 [m], R=15 [km]']

for k in ks:
    plt.scatter(rays[k].ts[:,-1].T, -rays[k].zs[:,-1].T, lw=0, s=10, label=labels[k])
plt.legend(markerscale=5)

plt.ylim([5000,-75])
plt.xlabel('time [s]')
plt.ylabel('depth [m]')

In [ ]:
fctd_None = fctd = tools.compute_filtered_fctd_grid(z_smooth=None, r_smooth=None)
fctdsmoothed = fctd = tools.compute_filtered_fctd_grid(z_smooth=200, r_smooth=15000)

In [ ]:
plt.figure(figsize=(13,4))

plt.subplot(1,2,1)
fctd_None['c'].assign_coords({'range':fctd_None.range/1000}).plot(cmap=cmocean.cm.thermal)
plt.xlim([-35, 120])
plt.ylim([3000,0])
plt.title('FCTD - no smoothing')
plt.xlabel('range [km]')
plt.ylabel('depth [m]')


plt.subplot(1,2,2)
fctdsmoothed['c'].assign_coords({'range':fctd_None.range/1000}).plot(cmap=cmocean.cm.thermal)
plt.xlim([-35, 120])
plt.ylim([3000,0])
plt.title('FCTD - $z_s$ = 200 [m], $r_s$ = 15 [km]')
plt.xlabel('range [km]')
plt.ylabel('depth [m]')
plt.tight_layout()


In [ ]:
plt.figure(figsize=(13,4))

plt.subplot(1,2,1)
fctd_None['c'].differentiate('depth').assign_coords({'range':fctd_None.range/1000}).plot(cmap=cmocean.cm.curl, cbar_kwargs={'label':'dc/dz [1/s]'})
plt.xlim([-35, 120])
plt.ylim([3000,0])
plt.title('FCTD - no smoothing')
plt.xlabel('range [km]')
plt.ylabel('depth [m]')


plt.subplot(1,2,2)
fctdsmoothed['c'].differentiate('depth').assign_coords({'range':fctd_None.range/1000}).plot(cmap=cmocean.cm.curl, cbar_kwargs={'label':'dc/dz [1/s]'})
plt.xlim([-35, 120])
plt.ylim([3000,0])
plt.title('FCTD - $z_s$ = 200 [m], $r_s$ = 15 [km]')
plt.xlabel('range [km]')
plt.ylabel('depth [m]')
plt.tight_layout()

In [ ]:
plt.figure(figsize=(13,4))

plt.subplot(1,2,1)
fctd_None['c'].differentiate('range').assign_coords({'range':fctd_None.range/1000}).plot(cmap=cmocean.cm.curl, cbar_kwargs={'label':'dc/dr [1/s]'})
plt.xlim([-35, 120])
plt.ylim([3000,0])
plt.title('FCTD - no smoothing')
plt.xlabel('range [km]')
plt.ylabel('depth [m]')


plt.subplot(1,2,2)
fctdsmoothed['c'].differentiate('range').assign_coords({'range':fctd_None.range/1000}).plot(cmap=cmocean.cm.curl, cbar_kwargs={'label':'dc/dr [1/s]'})
plt.xlim([-35, 120])
plt.ylim([3000,0])
plt.title('FCTD - $z_s$ = 200 [m], $r_s$ = 15 [km]')
plt.xlabel('range [km]')
plt.ylabel('depth [m]')
plt.tight_layout()

In [ ]:
fctd200['c'].plot()
plt.xlim([-35e3, 120e3])
plt.ylim([3000,0])

## Look at FCTD spectra

In [ ]:
import xrsignal as xrs

In [ ]:
fctd_spectra_z = xrs.welch(fctd_None.chunk(), dim='depth', fs=1, nperseg=512, dB=False, scaling='spectrum')

In [ ]:
_ = plt.plot(fctd_spectra_z.depth_frequency, 10*np.log10(fctd_spectra_z['c'].isel({'depth':0}).values), c='k', alpha=0.01)
plt.xlabel('Vertical Wavenumber [cpm]')

plt.xscale('log')
plt.ylabel('Spectral Density [dB rel $m^2$]')

plt.xlim([1e-3, 5e-1])
plt.ylim([-100,15])
plt.title('FCTD - Vertical Spectrum')
plt.vlines(1/200, -150,30, label='Filter Cutoff = 200 [m]')

plt.legend(loc='lower left')

In [ ]:
m,fctd_Sxx_r = signal.welch(fctd_None.isel({'range':slice(2,-2)})['c'].values, fs=1/500, nperseg=512, axis=1, scaling='spectrum')

In [ ]:
_ = plt.plot(m,10*np.log10(fctd_Sxx_r[:1000,:].T), c='k', alpha=0.01)

plt.xscale('log')

plt.ylim([-80, 35])
plt.xlim([1e-6, 1e-3])
plt.vlines(1/15000, -150,50, label='Filter Cutoff = 15 [km]')
plt.xlabel('Horizontal Wavenumber k [cpm]')
plt.ylabel('Horizontal Spectrum [dB rel $m^2$]')

plt.legend(loc='lower left')
plt.title('FCTD - Horizontal Spectrum: upper 1000 [m]')